In [ ]:
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

models = ['ItemKNN', 'BPR', 'NeuMF', 'GCMC']
datasets = ['ml-100k', 'jester', 'amazon_all_beauty', 'douban', 'netflix']
datatypes = ['expR', 'impR', 'impB']  
MovieLens, Jester, Amazon, Douban, Netflix = range(5)
ItemKNN, BPR, NeuMF, GCMC = range(4)
expR, impR, impB = range(3)

metrics = ['Recall@10', 'Precision@10', 'MRR@10', 'Hit@10', 'NDCG@10'] + \
            ['Recall@20', 'Precision@20', 'MRR@20', 'Hit@20', 'NDCG@20']
 
columns = ['Model', 'Dataset', 'Datatype'] + metrics + ['Epochs', 'Train', 'Eval', 'Params', 'FLOPs', 'CPU', 'GPU', 'MEM']
df = pd.DataFrame(columns=columns)

read_path = 'data/log/'
save_path = 'data/result/'

MODEL_COLORS = {
    'ItemKNN': '#f1c40f',  # Yellow
    'BPR':     '#2ecc71',  # Green
    'NeuMF':   '#e74c3c',  # Red
    'GCMC':    '#3498db',  # Blue
}

def read_log(log_path):
    training_data = {} # To store metric values
    test_data = {} # To store test result summary

    train_time = 0.0 # To store accumulated training time (in seconds)
    eval_time = 0.0 # To store accumulated eval time (in seconds)

    trainable_params = None  # To store "Trainable parameters: xxxxxxxx"
    flops = None             # To store "FLOPs: xxxxxx.xx"

    cpu_usage = None
    gpu_usage = None
    mem_usage = None

    with open(log_path, "r") as f:
        for line in f:
            # Extracting per-epoch metrics (mrr@10, recall@10, precision@10, etc.)
            metric_pattern = r"(\w+@\d+)\s*:\s*([\d.]+)"
            matches = re.findall(metric_pattern, line)
            for metric, value in matches:
                if metric not in training_data:
                    training_data[metric] = []
                training_data[metric].append(float(value))

            # Extract per-epoch Training Loss. 
            # Example: epoch 0 training [time: 282.57s, train loss: 325.2666]
            m_loss = re.search(r"train loss:\s*([\d.]+)", line)
            if m_loss:
                if 'loss' not in training_data:
                    training_data["loss"] = []
                training_data['loss'].append(float(m_loss.group(1)))

            # Extract Trainable Parameters. 
            # Example: Trainable parameters: 168128
            params_match = re.search(r"Trainable parameters:\s*([\d]+)", line)
            if params_match:
                trainable_params = int(params_match.group(1))

            # Extract FLOPs. 
            # Example: Tue 18 Nov 2025 23:07:57 INFO  FLOPs: 128.0
            flops_match = re.search(r"FLOPs:\s*([\d.]+)", line)
            if flops_match:
                flops = float(flops_match.group(1))

            # Accumulate training and evaluation time from epoch logs
            m_train = re.search(r"epoch\s+(\d+)\s+training\s+\[time:\s*([\d.]+)s", line)
            if m_train:
                train_time += float(m_train.group(2))

            m_eval = re.search(r"epoch\s+(\d+)\s+evaluating\s+\[time:\s*([\d.]+)s", line)
            if m_eval:
                eval_time += float(m_eval.group(2))

            # Match summary lines containing test results
            if "INFO  test result:" in line:
                test_data = {m[0]: float(m[1] or m[2]) for m in re.findall(r"\('([^']+)', (?:np\.float64\(([\d.]+)\)|([\d.]+))\)", line)}

            m_cpu = re.search(r"\|\s*CPU\s*\|\s*([\d.]+)\s*%", line)
            if m_cpu:
                cpu_usage = float(m_cpu.group(1))

            # GPU: capture used and total separately
            m_gpu = re.search(r"\|\s*GPU\s*\|\s*([\d.]+)\s*G\s*/\s*([\d.]+)\s*G", line)
            if m_gpu:
                gpu_usage = round(100.0 * float(m_gpu.group(1)) / float(m_gpu.group(2)), 2)

            # Memory: capture used and total separately
            m_mem = re.search(r"\|\s*Memory\s*\|\s*([\d.]+)\s*G\s*/\s*([\d.]+)\s*G", line)
            if m_mem:
                mem_usage = round(100.0 * float(m_mem.group(1)) / float(m_mem.group(2)), 2)

    epochs = len(next(iter(training_data.values())))  # Get the number of epochs

    return training_data, test_data, train_time, eval_time, epochs, trainable_params, flops, cpu_usage, gpu_usage, mem_usage

def normalize_loss(loss):
    # relative loss
    loss = np.array(loss)
    return (loss / loss[0])

def plot_4models(dataset, datatype, models_list, training_data_list, save_path):
    """
    Top:    Relative training loss
    Bottom: NDCG@10
    One color per model (same color in both plots)
    """

    fig, (ax_loss, ax_ndcg) = plt.subplots(
        2, 1, figsize=(10, 7), sharex=True
    )

    for model, training_data in zip(models_list, training_data_list):
        epochs = range(len(training_data['loss']))
        color = MODEL_COLORS[model]

        # ---------- LOSS (TOP) ----------
        loss_rel = normalize_loss(training_data['loss'])
        ax_loss.plot(
            np.array(epochs)+1,
            loss_rel,
            marker='v',
            linestyle='--',
            color=color,
            label=model
        )

        # ---------- NDCG (BOTTOM) ----------
        ax_ndcg.plot(
            np.array(epochs)+1,
            training_data['ndcg@10'],
            marker='.',
            linestyle='--',
            color=color,
            label=model
        )

        # mark best NDCG epoch
        best_ep = max(epochs, key=lambda i: training_data['ndcg@10'][i])
        ax_ndcg.plot(
            np.array(best_ep)+1,
            training_data['ndcg@10'][best_ep],
            marker='o',
            markersize=9,
            markerfacecolor='none',
            markeredgewidth=1.2,
            color=color
        )

    # ----- Labels & limits -----
    ax_loss.set_ylabel("Relative loss " + r"$\frac{\mathcal{L}}{\mathcal{L}(0)}$")
    ax_ndcg.set_ylabel("NDCG@10")
    ax_ndcg.set_xlabel("Epoch")

    ax_loss.set_ylim(0, 1.05)
    ax_ndcg.set_ylim(0, 1.05)
    ax_ndcg.set_xlim(0, 51)

    # ax_loss.grid(True)
    # ax_ndcg.grid(True)
    ###############################
    X_major_ticks = np.arange(0, 51, 10)
    X_minor_ticks = np.arange(0, 51, 1)
    Y_major_ticks = np.arange(0, 1.05, 0.1)
    Y_minor_ticks = np.arange(0, 1.05, 0.05)

    ax_loss.set_yticks(Y_major_ticks)
    ax_loss.set_yticks(Y_minor_ticks, minor=True)
    ax_ndcg.set_yticks(Y_major_ticks)
    ax_ndcg.set_yticks(Y_minor_ticks, minor=True)
    ax_ndcg.set_xticks(X_major_ticks)
    ax_ndcg.set_xticks(X_minor_ticks, minor=True)

    # different settings for the grids:
    # ax.grid(which='both')
    # ax.grid(which='major', alpha=0.2, color='#CCCCCC', linestyle='--')
    ax_loss.grid(which='minor', alpha=0.1)
    ax_loss.grid(which='major', alpha=0.5)
    ax_ndcg.grid(which='minor', alpha=0.1)
    ax_ndcg.grid(which='major', alpha=0.5)
    ###############################

    # ----- Legends (model-only, clean) -----
    ax_loss.legend(fontsize=9, loc="lower right")#, title="Model")
    ax_ndcg.legend(fontsize=9, loc="upper right")#, title="Model")

    fig.suptitle(
        f"Learning curves\nDataset: {dataset}  |  Datatype: {datatype}",
        y=0.92
    )

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"{save_path}/{dataset}--{datatype}.png", bbox_inches='tight')
    plt.close(fig)


def plot_all(save_path = 'data/result/', read_path = 'data/log/'):
    for t in [expR, impR, impB]:
        for d in [Amazon, MovieLens, Douban, Jester]:

            dataset = datasets[d]
            datatype = datatypes[t]

            models_list = []
            training_data_list = []

            for m in [ItemKNN, BPR, NeuMF, GCMC]:
                model = models[m]
                pattern = read_path + f'{model}-{dataset}--{datatype}-*.log'
                logs = glob.glob(pattern)

                if not logs:
                    continue

                training_data, *_ = read_log(logs[0])
                models_list.append(model)
                training_data_list.append(training_data)

            if len(training_data_list) > 0:
                plot_4models(dataset, datatype, models_list, training_data_list, save_path+'plots')


def table_csv(save_path = 'data/result/', read_path = 'data/log/'):
    df = pd.DataFrame(columns=columns)
    for t in [expR, impR, impB]:
        for d in [Amazon, MovieLens ,Douban, Jester]:
            for m in [ItemKNN, BPR, NeuMF, GCMC]:
                model = models[m]
                dataset = datasets[d]
                datatype = datatypes[t]
                # pattern: model--dataset--datatype-???.log
                log_read_pattern = read_path+f'{model}-{dataset}--{datatype}-*.log'
                for log in glob.glob(log_read_pattern):
                    training_data, test_data, train_time, eval_time, epochs, trainable_params, flops, cpu, gpu, mem = read_log(log)

                    # --- Test scores ---
                    row = {
                        'Model': model,
                        'Dataset': dataset,
                        'Datatype': datatype,
                        'Epochs': epochs,
                        'Train': round(train_time / epochs, 2),
                        'Eval': round(eval_time / epochs, 2),
                        'Params': trainable_params, 
                        'FLOPs': flops,
                        'CPU': cpu, 
                        'GPU': gpu, 
                        'MEM': mem,
                    }
                    # fill metrics (NaN if missing)
                    for x in metrics:
                        row[x] = test_data.get(x.lower(), float('NaN'))

                    df.loc[len(df)] = row  # append row
    df.to_csv(save_path+'df.csv', sep='\t', index=False)

plot_all()
table_csv()